In [ ]:
#!pip install easyocr

In [ ]:
import cv2
from PIL import Image
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import easyocr
import pandas as pd
import string
import math
import re
import glob
import ast

In [ ]:
class Text_From_Single_Image:

    def __init__(self, YOLO_result_path, YOLOimage_path, RAWimage_folder_path, output_path, filename):
        """
        Parameters:
        YOLO_result_path: str, path of floorplan_test-results.csv
        YOLOimage_path: str, path of specific image processed in YOLO model detection
        RAWimage_folder_path: str, path of folder containing raw images (high resolution), each image prefiex should find match with prefiex of image used in YOLO 
        output_path: str, file path for saving image annotated with text
        filename: str, name of image processed in YOLO model detection
        """
        self.YOLOimage_name = filename
        self.YOLOimage_path = YOLOimage_path
        self.RAWimage_path = self.find_RAWimage_path(RAWimage_folder_path)
        self.YOLO_result_path = YOLO_result_path
        self.raw_image = self.preprocess_image(show=False)
        self.yolo_result = self.initial_YOLO_result()
        self.text_result = self.read_text_from_image()
        self.target_image = self.draw_bounding_boxes(img_size_512=False)  
        self.target_image_size512 = self.draw_bounding_boxes(img_size_512=True)  
        self.image_result_path =  output_path.replace(".png", "_annoted.png") 
        self.image512_result_path =  output_path.replace(".png", "_512_annoted.png")

    def find_RAWimage_path(self, RAWimage_folder_path):
        def get_prefix_number(filename):
            match = re.match(r'^"?(\d+)', filename)  
            return match.group(1) if match else None

        base = get_prefix_number(self.YOLOimage_name)
        matches = glob.glob(os.path.join(RAWimage_folder_path, base + '*'))
        if matches:
            return matches[0]
        else:
            print(f"image is not found with prefix {base}")
            return None

    def preprocess_image(self, show=False):
        img = cv2.imread(self.RAWimage_path, cv2.IMREAD_GRAYSCALE)
        if show:
            plt.imshow(img)
            plt.title('Preprocessed')
            plt.axis('off')
            plt.show()
        return img

    def initial_YOLO_result(self, area_thres=0.5):
        df = pd.read_csv(self.YOLO_result_path)
        df = df[df['Image'] == self.YOLOimage_name]
        df = df[df['Confidence'] > area_thres]
        df = df[df['Class Name'] == 'area']
        df['BBox'] = df['BBox'].apply(ast.literal_eval)
        df['BBox'] = df['BBox'].apply(lambda x: [int(x[0]), int(x[1]), int(x[2]), int(x[3])])   
        return df
    
    def _normalize_bbox(self, bbox):
        return tuple((int(round(float(pt[0]))), int(round(float(pt[1])))) for pt in bbox)

    def _bbox_almost_equal(self, b1, b2, tol):
        for p1, p2 in zip(b1, b2):
            if math.hypot(p1[0]-p2[0], p1[1]-p2[1]) > tol:
                return False
        return True
    
    def read_text_from_image(self, threshold=0.25): 
        """
        Parameters:
            threshold: confidence threshold of text detection
        """
        area_df = self.yolo_result.copy()
        new_detections_horizontal = []
        new_detections_vertical = []
        
        for idx, row in area_df.iterrows():
            x_min, y_min, x_max, y_max = map(int, row['BBox'])
            YOLO_bbox = [x_min, y_min, x_max, y_max]
            cropped_img = self.raw_image[y_min:y_max, x_min:x_max] # Crop image
            
            reader = easyocr.Reader(['en'], gpu=False)
            detections_vertical = reader.readtext(cropped_img, rotation_info=[90, 270])
            detections_horizontal = reader.readtext(cropped_img)
            detections_vertical = [d for d in detections_vertical if d[2] >= threshold]
            detections_horizontal = [d for d in detections_horizontal if d[2] >= threshold]

            # Shift bbox back to original coordinates
            for bbox, text, conf in detections_horizontal:
                new_bbox = [[point[0] + x_min, point[1] + y_min] for point in bbox]
                new_detections_horizontal.append((new_bbox, text, YOLO_bbox, conf))
                
            for bbox, text, conf in detections_vertical:
                new_bbox = [[point[0] + x_min, point[1] + y_min] for point in bbox]
                new_detections_vertical.append((new_bbox, text, YOLO_bbox, conf))
            
        list_hor = [(item[0], item[1], item[2] if len(item) > 1 else None) for item in new_detections_horizontal]
        list_ver = [(item[0], item[1], item[2] if len(item) > 1 else None) for item in new_detections_vertical]

        map_h = {} #key:bbox, value:bbox,text,area
        for bbox, text, area in list_hor:
            key = self._normalize_bbox(bbox)
            if key in map_h:
                continue
            map_h[key] = [bbox, text, area]

        map_v = {} #key:bbox, value:bbox,text,area
        for bbox, text, area in list_ver:
            key = self._normalize_bbox(bbox)
            if key in map_v:
                continue
            map_v[key] = [bbox, text, area]

        merged = [] #final result
        processed_keys = set() # keep merged key record

        for key, (bbox_h, text_h, area) in map_h.items():
            # Find same detection exist in horizontal and vertical 
            if key in map_v:
                bbox_v, text_v, area = map_v[key]
                bbox = [[int(round(float(pt[0]))), int(round(float(pt[1])))] for pt in bbox_h]
                    
                # Keep the longer text (usually with higher accuracy)
                if(len(text_h) >= len(text_v)):
                    merged.append({"bbox": bbox, "text_h": text_h, "text_v": None, "Region":area})
                else:
                    merged.append({"bbox": bbox, "text_h": None, "text_v": text_v, "Region":area})
                processed_keys.add(key) 
            else:
                #get text detected in vertical direction but not in horizontal direction
                bbox = [[int(round(float(pt[0]))), int(round(float(pt[1])))] for pt in bbox_h]
                merged.append({"bbox": bbox, "text_h": text_h, "text_v": None, "Region":area})
                processed_keys.add(key)

        #get text detected in vertical direction but not in horizontal direction
        for key, (bbox_v, text_v, area) in map_v.items():
            if key not in processed_keys:
                bbox = [[int(round(float(pt[0]))), int(round(float(pt[1])))] for pt in bbox_v]
                merged.append({"bbox": bbox, "text_h": None, "text_v": text_v, "Region":area})

        #resize bbox to 512*512 graph in yolo
        original_h, original_w = self.raw_image.shape[:2]
        target_size = 512
        scale_x = target_size / original_w
        scale_y = target_size / original_h
                
        for detection in merged:
            original_bbox = detection['bbox']
            resized_bbox = []
            for point in original_bbox:
                new_x = int(point[0] * scale_x)
                new_y = int(point[1] * scale_y)
                resized_bbox.append([new_x, new_y])
                
            detection['bbox_512'] = resized_bbox
            detection['image'] = self.YOLOimage_name

        #convert result to dataframe
        def ocr_bbox_to_xyxy(bbox):
            try:
                bbox = ast.literal_eval(bbox)
            except:
                pass
            xs = [pt[0] for pt in bbox]
            ys = [pt[1] for pt in bbox]
            return [min(xs), min(ys), max(xs), max(ys)]
            
        merged_df = pd.DataFrame(merged)
        merged_df['text'] = merged_df['text_h'].combine_first(merged_df['text_v'])
        merged_df = merged_df.drop(columns=['text_h', 'text_v'])
        merged_df['bbox'] = merged_df['bbox'].apply(ocr_bbox_to_xyxy)
        merged_df['bbox_512'] = merged_df['bbox_512'].apply(ocr_bbox_to_xyxy)

        # Basic Cleaning and validation (only keep those having more than 1 character/digit)
        merged_df = merged_df[merged_df['text'].notna() &
                                      (merged_df['text'].str.len() > 1) &
                                      (merged_df['text'].str.contains(r'\w', regex=True))]

        # Filter Duplicate Detection in an image by IOU
        def calculate_iou(box1, box2):
            x1_1, y1_1, x2_1, y2_1 = box1
            x1_2, y1_2, x2_2, y2_2 = box2
                
            inter_x1 = max(x1_1, x1_2)
            inter_y1 = max(y1_1, y1_2)
            inter_x2 = min(x2_1, x2_2)
            inter_y2 = min(y2_1, y2_2)
                
            inter_width = max(0, inter_x2 - inter_x1)
            inter_height = max(0, inter_y2 - inter_y1)
            inter_area = inter_width * inter_height
            box1_area = (x2_1 - x1_1) * (y2_1 - y1_1)
            box2_area = (x2_2 - x1_2) * (y2_2 - y1_2)
            union_area = box1_area + box2_area - inter_area
        
            if union_area == 0:
                return 0.0
            return inter_area / union_area
        
        rows_to_keep = [True] * len(merged_df)
        
        for i in range(len(merged_df)):
            if not rows_to_keep[i]:
                continue
            
            box1 = merged_df.iloc[i]['bbox']
        
            # Compare with all subsequent rows
            for j in range(i + 1, len(merged_df)):
                if not rows_to_keep[j]:
                    continue
                
                box2 = merged_df.iloc[j]['bbox']
                iou = calculate_iou(box1, box2)
                if iou >= 0.85:
                    rows_to_keep[j] = False
                    
        merged_df = merged_df[rows_to_keep].reset_index(drop=True)
        return merged_df

    def draw_bounding_boxes(self, img_size_512=True):
        target_image = cv2.cvtColor(self.raw_image.copy(), cv2.COLOR_GRAY2BGR)
        if img_size_512:
            target_image = cv2.resize(target_image, (512, 512))
            
        for idx, detect in self.text_result.iterrows():
            if img_size_512:
                bbox = detect['bbox_512'] 
            else:
                bbox = detect['bbox']
            text = detect['text']

            top_left = (int(bbox[0]), int(bbox[1]))
            bottom_right = (int(bbox[2]), int(bbox[3]))
            
            cv2.rectangle(target_image, top_left,bottom_right, (0, 255, 0), 5)
            if img_size_512:
                cv2.putText(target_image, text, top_left, cv2.FONT_HERSHEY_PLAIN, 0.5, (0, 255, 0), 1)
            else:
                cv2.putText(target_image, text, top_left, cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 4)
        return target_image
    
    def show_image(self):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
        ax1.imshow(cv2.cvtColor(self.target_image, cv2.COLOR_BGR2RGB))
        ax1.set_title('Original Image')
        ax1.axis('off')
        
        ax2.imshow(cv2.cvtColor(self.target_image_size512, cv2.COLOR_BGR2RGB))
        ax2.set_title('Resized Image (512x512)')
        ax2.axis('off')
        plt.tight_layout()
        plt.show()

    def save(self):
        cv2.imwrite(self.image_result_path, self.target_image)
        cv2.imwrite(self.image512_result_path, self.target_image_size512)

In [ ]:
class Image_Text_Folder_Processor:
    def __init__(self, YOLO_result_path, YOLOimage_folder_path, RAWimage_folder_path, output_folder_path, 
                save_annotated_image=False, detail_text_result_provided=False, detail_text_result_path=None):
        """
        Parameters:
        YOLO_result_path: str, path of floorplan_test-results.csv
        YOLOimage_folder_path: str, path of folder containing images processed in YOLO model detection
        RAWimage_folder_path: str, path of folder containing raw images (high resolution), each image prefiex should find match with prefiex of image used in YOLO 
        output_folder_path: str, folder path for saving csv results of text annotated
        save_annotated_image: flag, indicates if image annotated image is needed
        detail_text_result_provided: flag, indicates if the floorplan_test-detail_text_result.csv already generated
        text_result_path: str, provide path of floorplan_test-detail_text_result.csv if have detail_text_result_provided is True

        Return/Save:
        floorplan_test-detail_text_result.csv: a table storing positions of detected text, and Region and image they belong (columns: bbox,bbox_512,image,text,Region)
        floorplan_test-text_result.csv: a table storing image name, regions, 
                                        and text and area (unit removed) detected in that region (columns: image,Region,text,area)
        """
        self.YOLO_result_path = YOLO_result_path
        self.YOLOimage_folder_path = YOLOimage_folder_path
        self.RAWimage_folder_path = RAWimage_folder_path
        self.output_folder_path = output_folder_path
        self.mid_text_result_provided = detail_text_result_provided
        self.save_annotated_image = save_annotated_image
        self.images: List[ImageProcessor] = []
        self.YOLO_result = self.initial_YOLO_result()
        self.text_result = self.initial_text_result(detail_text_result_path)
        self.final_result = self.combine_result()

    def initial_text_result(self, detail_text_result_path):
        if self.mid_text_result_provided:
            text_result = pd.read_csv(detail_text_result_path)
        else:
            text_result = self.Load_folder()
        try:
            text_result['Region'] = text_result['Region'].apply(ast.literal_eval)
        except:
            pass
        return text_result
                                        
    def initial_YOLO_result(self, area_thres=0.5):
        df = pd.read_csv(self.YOLO_result_path)
        df = df[df['Confidence'] > area_thres]
        df = df[df['Class Name'] == 'area']
        try:
             df['BBox'] =  df['BBox'].apply(ast.literal_eval)
        except:
            pass
        df['BBox'] = df['BBox'].apply(lambda x: [int(x[0]), int(x[1]), int(x[2]), int(x[3])])
        return df
        
    def Load_folder(self):
        if self.mid_text_result_provided:
            print("You already provide a detail text extraction result")
            return
        else:
            text_result = pd.DataFrame(columns=['bbox','bbox_512','image','text','Region'])
            for filename in os.listdir(self.YOLOimage_folder_path):
                YOLOimage_file_path = os.path.join(self.YOLOimage_folder_path, filename)
                output_file_path = os.path.join(self.output_folder_path, filename)
                image_object = Text_From_Single_Image(self.YOLO_result_path, YOLOimage_file_path, 
                                          self.RAWimage_folder_path, output_file_path, filename)
                self.images.append(image_object)
                if self.save_annotated_image:
                    image_object.save()
                text_result = pd.concat([text_result, image_object.text_result], ignore_index=True)
            return text_result

    def combine_result(self):
        final_result = []
        for image in (self.text_result['image'].unique()):
            img_yolo_df = self.YOLO_result[self.YOLO_result['Image'] == image].reset_index()
            img_text_df = self.text_result[self.text_result['image'] == image].reset_index()

            # Match text with area by YOLO_area column in img_text_df
            for index, area in img_yolo_df.iterrows():
                text = []
                number = []
                mask = img_text_df['Region'].apply(lambda x: x == area['BBox'])
                img_text_area_df = img_text_df[mask].reset_index(drop=True)

                for index, content in img_text_area_df.iterrows():
                    #if text before first space are number, remove character and store as number
                    text_before_space = content['text'].split()[0] if content['text'].strip() else ''
                    cleaned_text = re.sub(r'[^\d.]', '', text_before_space.replace(',', '.'))
                    if cleaned_text:
                        try:
                            number.append(float(cleaned_text))
                        except ValueError:
                                text.append(content['text'])
                    else:
                        text.append(content['text']) 
                        
                final_result.append({"image": image,"Region": area['BBox'], "text": text, "area": number})
        final_result = pd.DataFrame(final_result)
        return final_result

    def save_text_result(self):
        os.makedirs(self.output_folder_path, exist_ok=True)
        detail_path = os.path.join(self.output_folder_path, "floorplan_test-detail_text_result.csv")
        final_path = os.path.join(self.output_folder_path, "floorplan_test-text_result.csv")
        
        self.text_result.to_csv(detail_path, index=False)
        self.final_result.to_csv(final_path, index=False)

In [ ]:
csv_input_path = ".. .. /Floorplan/YOLO_result/floorplan_test-results.csv" # Add path of csv result from YOLO
YOLOimage_folder_path = ". . /Floorplan/dataset/images/test" # Add path of test image
RAWimage_folder_path = ". . /Floorplan/dataset/original_images/test" # Add path of original image
output_folder_path = ". . /Floorplan/text_result" #Add output path

FloorPlans = Image_Text_Folder_Processor(csv_input_path, 
                                         YOLOimage_folder_path, 
                                         RAWimage_folder_path, 
                                         output_folder_path, 
                                         save_annotated_image=True,
                                         detail_text_result_provided=False, 
                                         detail_text_result_path=None)

FloorPlans.save_text_result()

In [ ]:
FloorPlans.final_result